# 4. Build the Agent Loop Yourself

An **agent** is an LLM that has access to tools and runs in a loop until it finishes the assigned task.

The LLM does the thinking and deciding. Your Python code runs the loop and controls which tools are actually allowed to run.

A simple agent loop looks like this:

```text
task -> LLM -> tool call -> Python runs tool -> result goes back to LLM -> done or keep going
```


## Today's Goal

Build one small agent that can check where the International Space Station is right now and turn that live data into a mission-control style update.

The task is finished when the model has enough information to give the final answer.


In [1]:
import json
import os
from pprint import pprint

import requests
# If you do not have python-dotenv installed, run: pip install python-dotenv
from dotenv import load_dotenv

TIMEOUT = 60
MODEL = "gpt-4.1-mini"

## Load the API Key

This is the same setup from notebook 3. The real key lives in `.env`.


In [2]:
load_dotenv()

api_key = os.environ["OPENAI_API_KEY"]

print("OPENAI_API_KEY loaded")

OPENAI_API_KEY loaded


In [3]:
openai_url = "https://api.openai.com/v1/responses"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json",
}

## Step 1: Write a Tool

A tool starts as a normal Python function.

This one calls two public APIs: one to track the International Space Station, and one to turn its latitude and longitude into a readable place name. No extra API key is needed.


In [4]:
def get_place_name(latitude, longitude):
    reverse_geocode_url = "https://api.bigdatacloud.net/data/reverse-geocode-client"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "localityLanguage": "en",
    }
    response = requests.get(reverse_geocode_url, params=params, timeout=TIMEOUT)
    response.raise_for_status()
    data = response.json()

    place_parts = [
        data.get("locality") or data.get("city"),
        data.get("principalSubdivision"),
        data.get("countryName"),
    ]
    place_name = ", ".join(part for part in place_parts if part)

    if place_name:
        return place_name

    return "an area with no nearby place name"


def get_iss_location():
    iss_url = "https://api.wheretheiss.at/v1/satellites/25544"
    response = requests.get(iss_url, timeout=TIMEOUT)
    response.raise_for_status()
    data = response.json()

    latitude = data["latitude"]
    longitude = data["longitude"]

    return {
        "place_name": get_place_name(latitude, longitude),
        "latitude": round(latitude, 2),
        "longitude": round(longitude, 2),
        "altitude_km": round(data["altitude"], 2),
        "velocity_km_per_hour": round(data["velocity"], 2),
    }


In [5]:
pprint(get_iss_location())


{'altitude_km': 418.86,
 'latitude': 28.9,
 'longitude': -154.07,
 'place_name': 'Pacific Ocean',
 'velocity_km_per_hour': 27592.26}


## Step 2: Describe the Tool to the Model

The model cannot see our Python function. It only sees this tool description.

The description helps the model decide when the tool would be useful.

This tool has no arguments, so `properties` and `required` are empty.


In [6]:
tools = [
    {
        "type": "function",
        "name": "get_iss_location",
        "description": (
            "Get the current place name, latitude, longitude, altitude, and speed of the "
            "International Space Station. Use this for live ISS tracking questions."
        ),
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False,
        },
        "strict": True,
    }
]


## Step 3: Start the Conversation

The `messages` list is the conversation history we send to the model.

At first, it only has the user's task. Notice that we are not telling the model which tool to use here. The model should decide that from the tool description.


In [10]:
messages = [
    {
        "role": "user",
        "content": (
            "Where is the International Space Station right now? "
            "Give me a short mission-control update with its current place name, "
            "coordinates, altitude, and speed. Make it exciting but factual."
        ),
    }
]


## Step 4: Run the Agent Loop

This is the whole pattern in one cell.

If the model asks for the tool, Python runs it and adds the result to `messages`.

If the model returns normal text, the task is done and the loop stops.


In [11]:
turn = 0
while True:
    print("="*100)
    turn += 1
    print(f"Turn {turn}")
    pprint(messages)
    print("="*100)
    response = requests.post(
        openai_url,
        headers=headers,
        json={
            "model": MODEL,
            "input": messages,
            "tools": tools,
            "max_output_tokens": 300,
        },
        timeout=TIMEOUT,
    )
    response.raise_for_status()
    response_data = response.json()
    print("*"*100)
    pprint(response_data)
    print("*"*100)

    output = response_data["output"][0]
    print("Output type:", output["type"])

    if output["type"] == "function_call":
        print("Tool:", output["name"])

        if output["name"] == "get_iss_location":
            result = get_iss_location()
        else:
            raise ValueError(f"Unknown tool: {output['name']}")

        pprint(result)

        messages.append(output)
        messages.append(
            {
                "type": "function_call_output",
                "call_id": output["call_id"],
                "output": json.dumps(result),
            }
        )
    else:
        pprint(output["content"][0]["text"])
        break


Turn 1
[{'content': 'Where is the International Space Station right now? Give me a '
             'short mission-control update with its current place name, '
             'coordinates, altitude, and speed. Make it exciting but factual.',
  'role': 'user'}]
****************************************************************************************************
{'background': False,
 'billing': {'payer': 'developer'},
 'completed_at': 1781991422,
 'created_at': 1781991421,
 'error': None,
 'frequency_penalty': 0.0,
 'id': 'resp_0b88bbc30e36c687006a3707fdd3288199a22a594bfe9238b0',
 'incomplete_details': None,
 'instructions': None,
 'max_output_tokens': 300,
 'max_tool_calls': None,
 'metadata': {},
 'model': 'gpt-4.1-mini-2025-04-14',
 'moderation': None,
 'object': 'response',
 'output': [{'arguments': '{}',
             'call_id': 'call_sUSopqZgyv1QWAIuJo5gMHNZ',
             'id': 'fc_0b88bbc30e36c687006a3707feb178819997d6fd52a45d7c5c',
             'name': 'get_iss_location',
          

## What Happened

Python and the model each had a job:

- The model decided it needed current ISS data.
- Python called the public ISS tracking API.
- Python used latitude and longitude to look up a readable place name.
- Python sent the tool result back to the model.
- The model wrote the final mission update.

That loop is what makes this an agent instead of a single prompt.


## Quick Review

- A **tool** is a Python function the model is allowed to ask for.
- A **function call** is the model saying, "Please run this tool."
- A **function call output** is Python sending the result back.
- An **agent loop** keeps going until the model gives a final answer.
- The model can choose the tool, but your code controls what actually runs.
